# Wiki / Memory — Agent-Analyse

Tier-1 eval (#387). Tests `wiki` page-creation, recall-after-save,
transient-filter (ADR-020). Scoring via `expected.wiki:` evaluator.

## 1. Setup

In [ ]:
import os, sys, importlib
from pathlib import Path

PYTF_ROOT = Path.cwd()
if PYTF_ROOT.name == 'notebooks':
    PYTF_ROOT = PYTF_ROOT.parent
os.chdir(PYTF_ROOT)
for p in [str(PYTF_ROOT / 'src'), str(PYTF_ROOT / 'notebooks')]:
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    from dotenv import load_dotenv
    load_dotenv(PYTF_ROOT / '.env')
except ImportError:
    pass

try:
    import nest_asyncio; nest_asyncio.apply()
except ImportError:
    pass

import logging, structlog
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s | %(message)s')
structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))

import analysis_lib
importlib.reload(analysis_lib)
import analysis_lib as alib
print('analysis_lib OK -', len(alib.list_evaluators()), 'evaluators')

## 2. Agent bauen (wiki-only)

Tools: `wiki`, `web_search`, `file_read`. Test der Page-Naming-Konvention, Recall, und Transient-Filter (ADR-020).

In [ ]:
WORKDIR = Path('.taskforce_wiki_analysis')
WORKDIR.mkdir(exist_ok=True)
WIKI_TOOLS = ['wiki', 'web_search', 'file_read']
WIKI_PROMPT = (
    'Du bist ein persönlicher Assistent mit Wiki-Gedächtnis (`.taskforce/memory/wiki/`). '
    'Persistiere reusable Info (Präferenzen, Kontakte, Entitäten) als markdown-Pages '
    'unter logischen Pfaden: entities/<slug>.md für Personen, preferences/<topic>.md '
    'für Vorlieben. NIEMALS transiente Werte (Uhrzeit, einmalige Tasks) ins Wiki schreiben. '
    'Bei Recall-Fragen via wiki(search) suchen.'
)

async def build_wiki_agent():
    a = await factory.create_agent(
        system_prompt=WIKI_PROMPT, tools=WIKI_TOOLS,
        persistence={'type': 'file', 'work_dir': str(WORKDIR)},
        work_dir=str(WORKDIR),
        planning_strategy='native_react', max_steps=10,
    )
    alib.patch_anti_compression(a, summary_threshold=80, tool_result_store_threshold=6000)
    return a, len(WIKI_PROMPT)

BUILD_AGENT = build_wiki_agent
_smoke, _chars = await BUILD_AGENT()
AVAILABLE_TOOLS = set(_smoke.tools.keys())
print(f'tools={sorted(AVAILABLE_TOOLS)}  prompt={_chars} chars')
await _smoke.close()

## 3. Szenarien laden

Aus `scenarios/memory_wiki.yaml`.

In [ ]:
all_scenarios = alib.load_scenarios('notebooks/scenarios/memory_wiki.yaml')
eligible = alib.filter_scenarios(all_scenarios, AVAILABLE_TOOLS)
print(f'Total: {len(all_scenarios)}, Eligible: {len(eligible)}')
for s in eligible:
    print(f'  - {s.id:35s} ({s.category}/{s.difficulty})')

## 4. Einzelnes Szenario (Detail-Lauf)

In [ ]:
TARGET_ID = 'wiki-save-contact-naming'  # change me
target = next((s for s in eligible if s.id == TARGET_ID), None) or eligible[0]
print(f'Target: {target.id}')
print(f'Mission: {target.mission}')
print(f'Hidden intent: {target.hidden_intent}')

In [ ]:
agent, sys_chars = await BUILD_AGENT()
rec = await alib.run(
    executor, agent, target.mission,
    project_root=WORKDIR, snapshot_subdirs=('memory', 'memory/wiki'),
    initial_system_prompt_chars=sys_chars, silent=False,
)
# Stash MCP tool names if any (no-op for non-MCP notebooks)
rec.extra['mcp_tool_names'] = list(_mcp_tool_names_of(agent)) if '_mcp_tool_names_of' in dir() else []
alib.print_summary(rec)

## 5. Reports

In [ ]:
alib.print_tool_stats(rec)
print()
alib.print_tool_results(rec, head=15)

In [ ]:
card = alib.score_rule_based(rec, target)
print(f'=== Rule-based ({"PASS" if card.passed else "FAIL"}) ===')
alib.print_feature_checks(card)
if card.details:
    print('Details:')
    for d in card.details:
        print(f'  - {d}')

In [ ]:
await agent.close()
_ = alib.plot_tool_frequencies(rec, title=f'Tool-Aufrufe: {target.id}')

## 6. Batch-Lauf

In [ ]:
results = await alib.run_scenarios(
    executor, BUILD_AGENT, eligible,
    project_root=WORKDIR, snapshot_subdirs=('memory', 'memory/wiki'),
    reset_dirs_before_each=('memory', 'memory/wiki'),
    repeats=1, progress=True,
)  
print()
alib.print_scenario_summary(results)

In [ ]:
_ = alib.plot_scenario_matrix(results, metric='passed', title='Pass/Fail')
_ = alib.plot_scenario_matrix(results, metric='tool_calls', title='Tool calls')

## Ideen für weitere Experimente

Siehe TODOs in der Scenario-YAML.

In [ ]:
print('done')